In [ ]:
import geopandas as gpd
import pandas as pd
from IPython.display import display
from scipy import stats

from analysis.src.constants import COMPONENTS
from analysis.src.io import experiment_paths, load_data
from paths import paths

In [ ]:
OUT = experiment_paths('index/validation')

data = load_data()

In [ ]:
# Load and join data
medsat = pd.read_csv(paths.raw.shared.medsat).set_index('geography code')
joined = data[['walkability_score']].join(medsat[[col for col in medsat.columns if 'c_percent commute' in col]])

In [ ]:
# Calculate Spearman correlation coefficients
car = joined[['walkability_score', 'c_percent commute car']].dropna()
corr_car, p_value_car = stats.spearmanr(car.walkability_score, car['c_percent commute car'])

walk = joined[['walkability_score', 'c_percent commute on foot']].dropna()
corr_walk, p_value_walk = stats.spearmanr(walk.walkability_score, walk['c_percent commute on foot'])

walk_bike = joined[['walkability_score']].copy()
walk_bike['commute_share'] = joined['c_percent commute on foot'] + joined['c_percent commute bicycle']
walk_bike = walk_bike.dropna()
corr_walk_bike, p_value_walk_bike = stats.spearmanr(walk_bike.walkability_score, walk_bike.commute_share)

print('Car', corr_car, p_value_car)
print('Walk', corr_walk, p_value_walk)
print('Walk + Bike', corr_walk_bike, p_value_walk_bike)

In [ ]:
# Load Stockton's 2001 OA scores and the 2021 LSOA boundaries
stockton_data = pd.read_csv(paths.raw.validation.stockton, usecols = ['oa_00_2001', 'OA_M1_raw'])
lsoas = data[['geometry']].reset_index().to_crs('EPSG:27700')

oa_boundaries = gpd.read_file(
    paths.raw.validation.boundaries, bbox = tuple(lsoas.total_bounds), columns = ['OA01CDOLD']
).to_crs(lsoas.crs)

oa_scores = oa_boundaries.merge(
    stockton_data, left_on = 'OA01CDOLD', right_on = 'oa_00_2001', validate = 'one_to_one'
)

In [ ]:
# Intersect the old OA polygons with the current LSOAs
intersections = gpd.overlay(
    lsoas, oa_scores[['OA_M1_raw', 'geometry']], how = 'intersection',
    keep_geom_type = True, make_valid = True
)
intersections['area'] = intersections.geometry.area
intersections = intersections.loc[intersections.area > 0].copy()

In [ ]:
# Average scores using the area of each OA inside each 2021 LSOA
intersections['weighted_score'] = intersections.OA_M1_raw * intersections.area
totals = intersections.groupby('LSOA21CD')[['weighted_score', 'area']].sum()
stockton_scores = (totals.weighted_score / totals.area).rename('reference')

coverage = lsoas.set_index('LSOA21CD').geometry.make_valid().area.to_frame('target_area_m2')
coverage['covered_area_m2'] = totals.area
coverage['coverage_fraction'] = coverage.covered_area_m2 / coverage.target_area_m2

In [ ]:
# Compare the composite index with the mapped Stockton scores
paired = data[['walkability_score']].join(stockton_scores).dropna()
corr_stockton, p_value_stockton = stats.spearmanr(paired.reference, paired.walkability_score)

print('Stockton correlation', corr_stockton, p_value_stockton)

validation_results = pd.DataFrame({
    'reference': ['Car', 'Walking', 'Walking and cycling', 'Stockton'],
    'n_lsoas': [len(car), len(walk), len(walk_bike), len(paired)],
    'spearman_rho': [corr_car, corr_walk, corr_walk_bike, corr_stockton],
    'p_value': [p_value_car, p_value_walk, p_value_walk_bike, p_value_stockton],
})

validation_results.to_csv(OUT.validation_csv, index = False)
paired.to_csv(OUT.stockton_matched_scores_csv, index_label = 'LSOA21CD')

display(validation_results)
display(coverage.coverage_fraction.describe())

In [ ]:
rows = []
for variable in [*COMPONENTS, 'walkability_score']:
    sample = data[[variable]].join(stockton_scores).dropna()
    spearman, p_spearman = stats.spearmanr(sample[variable], sample.reference)
    pearson, p_pearson = stats.pearsonr(sample[variable], sample.reference)

    rows.append({
        'variable': variable,
        'n_lsoas': len(sample),
        'spearman_rho': spearman,
        'spearman_p_value': p_spearman,
        'pearson_r': pearson,
        'pearson_p_value': p_pearson
    })

stockton_correlations = pd.DataFrame(rows)
stockton_correlations.to_csv(OUT.stockton_correlations_csv, index = False)

display(stockton_correlations)